# ScentAI Gemma 4 Full Run-All Notebook

This notebook is intentionally linear and safe for Colab **Runtime > Run all**.

It performs the main full LoRA training run, saves the adapter, runs quick inference, and writes the fixed ScentAI eval-v2 outputs.

Removed from this run-all version: old grounding-error-analysis cell, zip-output cell, and manual pilot/preflight decision cells.

Expected final eval files:
- `/content/drive/MyDrive/Perfume-Dataset/train_set/eval/runs/full_gemma4_12b_eval_v2_strictprompt_v2_outputs.jsonl`
- `/content/drive/MyDrive/Perfume-Dataset/train_set/eval/runs/full_gemma4_12b_eval_v2_strictprompt_v2_metadata.json`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Optional Hugging Face login. Put HF_TOKEN in Colab Secrets if the model requires auth.
import os
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print('Logged in to Hugging Face.')
else:
    print('HF_TOKEN not found. This is OK if the selected model is public in your environment.')


In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/Perfume-Dataset')
DATA_DIR = PROJECT_DIR / 'train_set' / 'finetune' / 'gemma'

# Modes: overfit -> tiny sanity, mini -> 500-example smoke, pilot -> 4.75K, full -> 30.4K
RUN_MODE = 'full'

# Main target. Gemma 4 has native system-role support and stronger reasoning than our Gemma 2 baseline.
MODEL_PRESET = 'gemma4_12b'
MODEL_PRESETS = {
    'gemma4_12b': {
        'model_name': 'google/gemma-4-12B-it',
        'output_slug': 'gemma-4-12b-it',
        'supports_native_system_role': True,
        'max_seq_length': 4096,
        'notes': 'Primary ScentAI target: Gemma 4 12B instruction-tuned with native system role.',
    },
    'gemma3_12b': {
        'model_name': 'unsloth/gemma-3-12b-it-bnb-4bit',
        'output_slug': 'gemma-3-12b-it',
        'supports_native_system_role': False,
        'max_seq_length': 4096,
        'notes': 'Fallback only if Gemma 4 access/runtime fails.',
    },
}
MODEL_CONFIG = MODEL_PRESETS[MODEL_PRESET]
MODEL_NAME = MODEL_CONFIG['model_name']
USE_NATIVE_SYSTEM_ROLE = MODEL_CONFIG['supports_native_system_role']
OUTPUT_DIR = PROJECT_DIR / 'models' / f"scentai-{MODEL_CONFIG['output_slug']}-{RUN_MODE}-fastmodel-lora"

# Set to a saved adapter path to continue from adapter weights after a Colab interruption.
# This resumes LoRA weights only, not optimizer/scheduler state.
RESUME_LORA_DIR = None

MAX_SEQ_LENGTH = 3072
LOAD_IN_4BIT = True
DISABLE_GEMMA_THINKING = True

# Stability-first profile for Gemma 4 12B on A100 80GB.
STABILITY_PROFILE = 'gemma4_12b_a100_safe'
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
USE_DORA = False
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GROUP_BY_LENGTH = True
GROUNDING_EVAL_LIMIT = 10
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

OVERFIT_TRAIN_EXAMPLES = 32
MINI_TRAIN_EXAMPLES = 500
MINI_VAL_EXAMPLES = 100
# Full validation every eval step is expensive. Use a representative subset during training;
# run a full validation pass separately after a promising run.
FULL_EVAL_EXAMPLES = 500

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

TRAIN_FILE = DATA_DIR / ('full_train.jsonl' if RUN_MODE == 'full' else 'pilot_train_4750.jsonl')
VAL_FILE = DATA_DIR / ('full_validation.jsonl' if RUN_MODE == 'full' else 'pilot_validation_250.jsonl')

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR   :', DATA_DIR)
print('TRAIN_FILE :', TRAIN_FILE, TRAIN_FILE.exists())
print('VAL_FILE   :', VAL_FILE, VAL_FILE.exists())
print('OUTPUT_DIR :', OUTPUT_DIR)
print('RUN_MODE   :', RUN_MODE)
print('MODEL_PRESET:', MODEL_PRESET)
print('MODEL_NAME :', MODEL_NAME)
print('MODEL_NOTES:', MODEL_CONFIG['notes'])
print('USE_NATIVE_SYSTEM_ROLE:', USE_NATIVE_SYSTEM_ROLE)
print('MAX_SEQ_LENGTH:', MAX_SEQ_LENGTH)
print('DISABLE_GEMMA_THINKING:', DISABLE_GEMMA_THINKING)
print('STABILITY_PROFILE:', STABILITY_PROFILE)
print('LoRA:', {'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT, 'use_dora': USE_DORA, 'targets': LORA_TARGET_MODULES})
print('Batch/grouping:', {'train_batch': PER_DEVICE_TRAIN_BATCH_SIZE, 'eval_batch': PER_DEVICE_EVAL_BATCH_SIZE, 'group_by_length': GROUP_BY_LENGTH})
print('GROUNDING_EVAL_LIMIT:', GROUNDING_EVAL_LIMIT)
print('RESUME_LORA_DIR:', RESUME_LORA_DIR)

if not TRAIN_FILE.exists() or not VAL_FILE.exists():
    raise FileNotFoundError('Fine-tune split files are missing. Upload train_set/finetune/gemma to Drive first.')


In [ ]:
# Run-all safety check: fail before expensive training if config is not full-run clean.
assert RUN_MODE == 'full', f'Expected RUN_MODE=full, got {RUN_MODE!r}'
assert MODEL_PRESET == 'gemma4_12b', f'Expected gemma4_12b, got {MODEL_PRESET!r}'
assert RESUME_LORA_DIR is None, f'Full run should start clean; got RESUME_LORA_DIR={RESUME_LORA_DIR!r}'
assert TRAIN_FILE.name == 'full_train.jsonl', TRAIN_FILE
assert VAL_FILE.name == 'full_validation.jsonl', VAL_FILE
print('FULL RUN-ALL CONFIG OK')
print('Output dir:', OUTPUT_DIR)


In [ ]:
# Gemma 4 12B-it uses model_type=gemma4_unified.
# The current pip release pulled by Unsloth may not recognize it yet, so install Transformers from source last.
# After this cell finishes, restart the Colab runtime once, then run the notebook from the top.
!pip -q install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install -U accelerate peft trl bitsandbytes sentencepiece datasets
!pip -q install -U --no-deps "git+https://github.com/huggingface/transformers.git"

import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print('transformers:', transformers.__version__)
print('gemma4_unified supported:', 'gemma4_unified' in CONFIG_MAPPING)
if 'gemma4_unified' not in CONFIG_MAPPING:
    raise RuntimeError('Transformers still does not support gemma4_unified. Restart runtime and rerun install, or Gemma 4 support has not installed correctly.')


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('CUDA count    :', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))


In [ ]:
import json
from datasets import load_dataset, DatasetDict

raw = load_dataset('json', data_files={'train': str(TRAIN_FILE), 'validation': str(VAL_FILE)})

if RUN_MODE == 'overfit':
    tiny_train = raw['train'].select(range(min(OVERFIT_TRAIN_EXAMPLES, len(raw['train']))))
    raw = DatasetDict({
        'train': tiny_train,
        # Overfit eval intentionally uses the same tiny examples.
        'validation': tiny_train,
    })
elif RUN_MODE == 'mini':
    raw = DatasetDict({
        'train': raw['train'].select(range(min(MINI_TRAIN_EXAMPLES, len(raw['train'])))),
        'validation': raw['validation'].select(range(min(MINI_VAL_EXAMPLES, len(raw['validation'])))),
    })
elif RUN_MODE == 'full' and FULL_EVAL_EXAMPLES:
    raw = DatasetDict({
        'train': raw['train'],
        'validation': raw['validation'].select(range(min(FULL_EVAL_EXAMPLES, len(raw['validation'])))),
    })

print(raw)
print(raw['train'][0].keys())
print([m['role'] for m in raw['train'][0]['messages']])
print(raw['train'][0]['messages'][1]['content'][:500])


In [ ]:
SYSTEM_OPEN = '[SYSTEM INSTRUCTIONS]'
SYSTEM_CLOSE = '[/SYSTEM INSTRUCTIONS]'

def render_gemma_user_answer(example):
    messages = example['messages']
    system = next((m['content'] for m in messages if m['role'] == 'system'), '').strip()
    user = next((m['content'] for m in messages if m['role'] == 'user'), '').strip()
    answer = next((m['content'] for m in messages if m['role'] in {'model', 'assistant'}), '').strip()

    if USE_NATIVE_SYSTEM_ROLE:
        user_content = user
        system_content = system
    else:
        # Fallback path for Gemma 3-style templates without native system-role support.
        user_content = f"{SYSTEM_OPEN}\n{system}\n{SYSTEM_CLOSE}\n\n{user}" if system else user
        system_content = ''

    return {'system_content': system_content, 'user_content': user_content, 'answer': answer}

train_text_ds = raw['train'].map(render_gemma_user_answer, remove_columns=raw['train'].column_names)
val_text_ds = raw['validation'].map(render_gemma_user_answer, remove_columns=raw['validation'].column_names)
print(train_text_ds)
print('system_content:', train_text_ds[0]['system_content'][:500])
print('user_content:', train_text_ds[0]['user_content'][:1200])
print()
print('--- ANSWER ---')
print(train_text_ds[0]['answer'][:800])


## >>> COPY/UPDATE CELL 1 / 5: `load-model`
**LOAD MODEL - processor/text tokenizer ayrimi burada**

Bu markdown başlığının hemen altındaki code cell’i komple kopyala/güncelle.


In [ ]:
# >>> COPY/UPDATE CELL 1 / 5: load-model
# === LOAD MODEL - processor/text tokenizer ayrimi burada

from pathlib import Path
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    unsloth_force_compile=False,
)

PROCESSOR = tokenizer
TEXT_TOKENIZER = getattr(PROCESSOR, 'tokenizer', PROCESSOR)
if getattr(TEXT_TOKENIZER, 'pad_token_id', None) is None:
    TEXT_TOKENIZER.pad_token = TEXT_TOKENIZER.eos_token
if getattr(tokenizer, 'pad_token_id', None) is None and hasattr(tokenizer, 'pad_token'):
    tokenizer.pad_token = getattr(TEXT_TOKENIZER, 'pad_token', None)

resume_dir = Path(RESUME_LORA_DIR) if RESUME_LORA_DIR else None
if resume_dir and resume_dir.exists():
    from peft import PeftModel

    model = PeftModel.from_pretrained(model, str(resume_dir), is_trainable=True)
    print('Loaded trainable LoRA adapter from:', resume_dir)
else:
    peft_kwargs = dict(
        r=LORA_R,
        target_modules=LORA_TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=20260629,
        use_rslora=False,
        loftq_config=None,
    )
    if USE_DORA:
        peft_kwargs['use_dora'] = True
    try:
        model = FastModel.get_peft_model(model, **peft_kwargs)
        dora_enabled = USE_DORA
    except (TypeError, ValueError) as exc:
        message = str(exc).lower()
        if not USE_DORA or ('use_dora' not in message and 'dora' not in message):
            raise
        print('DoRA is not supported by this Unsloth/FastModel path; retrying with vanilla LoRA.')
        peft_kwargs.pop('use_dora', None)
        model = FastModel.get_peft_model(model, **peft_kwargs)
        dora_enabled = False
    print('FastModel adapter ready:', {'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT, 'use_dora': dora_enabled})


def decode_generated_tokens(token_ids):
    # Official Gemma 4 examples decode via the processor, then parse_response.
    if hasattr(PROCESSOR, 'decode'):
        raw = PROCESSOR.decode(token_ids, skip_special_tokens=False)
        if hasattr(PROCESSOR, 'parse_response'):
            try:
                parsed = PROCESSOR.parse_response(raw)
                if isinstance(parsed, dict):
                    return parsed.get('text') or parsed.get('content') or str(parsed)
                if parsed is not None:
                    return str(parsed)
            except Exception:
                pass
        return raw.replace('<turn|>', '').strip()
    return TEXT_TOKENIZER.decode(token_ids, skip_special_tokens=True).strip()


## >>> COPY/UPDATE CELL 2 / 5: `preflight-template-tokenization`
**PREFLIGHT - Gemma 4 template/tokenization testi burada**

Bu markdown başlığının hemen altındaki code cell’i komple kopyala/güncelle.


In [ ]:
# >>> COPY/UPDATE CELL 2 / 5: preflight-template-tokenization
# === PREFLIGHT - Gemma 4 template/tokenization testi burada

# Preflight checks before dataset.map/training.
# Run this after load-model and before trainer. It should fail fast if Gemma 4 template/tokenization is incompatible.
import json
import torch

PROCESSOR = tokenizer
TEXT_TOKENIZER = getattr(PROCESSOR, 'tokenizer', PROCESSOR)
if getattr(TEXT_TOKENIZER, 'pad_token_id', None) is None:
    TEXT_TOKENIZER.pad_token = TEXT_TOKENIZER.eos_token

print('Preflight model:', MODEL_NAME)
print('Preflight processor:', type(PROCESSOR).__name__)
print('Preflight text tokenizer:', type(TEXT_TOKENIZER).__name__)
print('pad/eos:', TEXT_TOKENIZER.pad_token_id, TEXT_TOKENIZER.eos_token_id)
print('chat_template exists:', bool(getattr(PROCESSOR, 'chat_template', None)))

sample = train_text_ds[0]
messages = []
if USE_NATIVE_SYSTEM_ROLE and sample.get('system_content'):
    messages.append({'role': 'system', 'content': sample['system_content']})
messages.append({'role': 'user', 'content': sample['user_content']})
messages.append({'role': 'assistant', 'content': sample['answer']})

rendered = PROCESSOR.apply_chat_template(messages, add_generation_prompt=False, tokenize=False, enable_thinking=not DISABLE_GEMMA_THINKING)
answer_start = rendered.rfind(sample['answer'].strip())
print('rendered_len:', len(rendered), 'answer_start:', answer_start)
print('rendered_tail:', repr(rendered[-500:]))
if answer_start < 0:
    raise ValueError('Preflight failed: assistant answer cannot be located in rendered chat template.')

encoded = TEXT_TOKENIZER(rendered, add_special_tokens=False, return_offsets_mapping=True)
if 'offset_mapping' not in encoded:
    raise ValueError('Preflight failed: tokenizer did not provide offset_mapping.')
labels = [token_id if end > answer_start else -100 for token_id, (start, end) in zip(encoded['input_ids'], encoded['offset_mapping'])]
trainable = sum(1 for value in labels if value != -100)
print('token_count:', len(encoded['input_ids']), 'trainable_answer_tokens:', trainable)
if trainable <= 0:
    raise ValueError('Preflight failed: zero trainable answer tokens.')

# Tiny generation check with current chat template. This catches role/template issues before training.
test_messages = []
if USE_NATIVE_SYSTEM_ROLE:
    test_messages.append({'role': 'system', 'content': 'You are a concise assistant.'})
test_messages.append({'role': 'user', 'content': 'Say hello in one short sentence.'})
inputs = PROCESSOR.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
).to(next(model.parameters()).device)
with torch.inference_mode():
    out = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=TEXT_TOKENIZER.pad_token_id,
        eos_token_id=TEXT_TOKENIZER.eos_token_id,
    )
new_tokens = out[0][inputs['input_ids'].shape[-1]:]
print('generation_smoke:', repr(decode_generated_tokens(new_tokens)))
print('PRECHECK_OK')


## >>> COPY/UPDATE CELL 3 / 5: `trainer`
**TRAINER - response-only tokenization fix burada**

Bu markdown başlığının hemen altındaki code cell’i komple kopyala/güncelle.


In [ ]:
# >>> COPY/UPDATE CELL 3 / 5: trainer
# === TRAINER - response-only tokenization fix burada

from transformers import TrainingArguments, Trainer, TrainerCallback
import torch
import json
from datetime import datetime, timezone

PROCESSOR = tokenizer
TEXT_TOKENIZER = getattr(PROCESSOR, 'tokenizer', PROCESSOR)
if getattr(TEXT_TOKENIZER, 'pad_token_id', None) is None:
    TEXT_TOKENIZER.pad_token = TEXT_TOKENIZER.eos_token

def build_prompt_messages(example):
    messages = []
    system_content = example.get('system_content', '').strip()
    if USE_NATIVE_SYSTEM_ROLE and system_content:
        messages.append({'role': 'system', 'content': system_content})
    messages.append({'role': 'user', 'content': example['user_content']})
    return messages

def render_full_chat(example):
    answer = example['answer'].strip()
    full_text = PROCESSOR.apply_chat_template(
        build_prompt_messages(example) + [{'role': 'assistant', 'content': answer}],
        add_generation_prompt=False,
        tokenize=False,
        enable_thinking=not DISABLE_GEMMA_THINKING,
    )
    answer_start = full_text.rfind(answer)
    if answer_start < 0:
        preview = full_text[-1000:]
        raise ValueError(f'Could not locate assistant answer inside rendered chat template. Render tail: {preview!r}')
    return full_text, answer_start, answer

def tokenize_with_answer_mask(example):
    full_text, answer_start, answer = render_full_chat(example)
    encoded = TEXT_TOKENIZER(
        full_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    input_ids = encoded['input_ids']
    offsets = encoded.get('offset_mapping')
    if offsets is None:
        raise ValueError('Tokenizer did not return offset_mapping; a fast tokenizer is required for robust Gemma 4 masking.')

    labels = []
    for token_id, (start, end) in zip(input_ids, offsets):
        if end <= answer_start:
            labels.append(-100)
        else:
            labels.append(token_id)

    original_len = len(input_ids)
    original_answer_tokens = sum(1 for value in labels if value != -100)
    original_prompt_tokens = original_len - original_answer_tokens

    if original_len > MAX_SEQ_LENGTH:
        overflow = original_len - MAX_SEQ_LENGTH
        input_ids = input_ids[overflow:]
        labels = labels[overflow:]

    attention_mask = [1] * len(input_ids)
    answer_tokens = sum(1 for value in labels if value != -100)
    prompt_tokens = len(input_ids) - answer_tokens
    was_answer_truncated = answer_tokens < original_answer_tokens
    was_prompt_truncated = prompt_tokens < original_prompt_tokens

    if answer_tokens == 0:
        raise ValueError('Tokenization produced zero trainable answer tokens.')

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
        'prompt_tokens': prompt_tokens,
        'original_prompt_tokens': original_prompt_tokens,
        'answer_tokens': answer_tokens,
        'original_answer_tokens': original_answer_tokens,
        'was_prompt_truncated': was_prompt_truncated,
        'was_answer_truncated': was_answer_truncated,
    }

def tokenize_response_only(example):
    return tokenize_with_answer_mask(example)

class ResponseOnlyDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(feature['input_ids']) for feature in features)
        pad_id = getattr(self.tokenizer, 'pad_token_id', None)
        if pad_id is None and hasattr(self.tokenizer, 'tokenizer'):
            pad_id = self.tokenizer.tokenizer.pad_token_id
        batch = {'input_ids': [], 'attention_mask': [], 'labels': []}

        for feature in features:
            pad_len = max_len - len(feature['input_ids'])
            batch['input_ids'].append(feature['input_ids'] + [pad_id] * pad_len)
            batch['attention_mask'].append(feature['attention_mask'] + [0] * pad_len)
            batch['labels'].append(feature['labels'] + [-100] * pad_len)

        return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}

train_ds = train_text_ds.map(tokenize_response_only, remove_columns=train_text_ds.column_names)
val_ds = val_text_ds.map(tokenize_response_only, remove_columns=val_text_ds.column_names)
data_collator = ResponseOnlyDataCollator(tokenizer)

def summarize_tokenization(dataset, name):
    n = len(dataset)
    prompt_tokens = [int(row['prompt_tokens']) for row in dataset]
    original_prompt_tokens = [int(row['original_prompt_tokens']) for row in dataset]
    answer_tokens = [int(row['answer_tokens']) for row in dataset]
    prompt_truncated = sum(1 for row in dataset if row['was_prompt_truncated'])
    answer_truncated = sum(1 for row in dataset if row['was_answer_truncated'])
    return {
        'name': name,
        'count': n,
        'prompt_tokens_avg': sum(prompt_tokens) / max(n, 1),
        'prompt_tokens_max': max(prompt_tokens) if prompt_tokens else 0,
        'original_prompt_tokens_max': max(original_prompt_tokens) if original_prompt_tokens else 0,
        'answer_tokens_avg': sum(answer_tokens) / max(n, 1),
        'answer_tokens_max': max(answer_tokens) if answer_tokens else 0,
        'prompt_truncated_count': prompt_truncated,
        'prompt_truncated_ratio': prompt_truncated / max(n, 1),
        'answer_truncated_count': answer_truncated,
        'answer_truncated_ratio': answer_truncated / max(n, 1),
    }

tokenization_stats = {
    'run_mode': RUN_MODE,
    'model_preset': MODEL_PRESET,
    'model_name': MODEL_NAME,
    'use_native_system_role': USE_NATIVE_SYSTEM_ROLE,
    'max_seq_length': MAX_SEQ_LENGTH,
    'train': summarize_tokenization(train_ds, 'train'),
    'validation': summarize_tokenization(val_ds, 'validation'),
}
print('Tokenized train:', train_ds)
print('Tokenization stats:', json.dumps(tokenization_stats, indent=2))

if RUN_MODE == 'overfit':
    max_steps = 80
    num_train_epochs = 1
    save_steps = 40
    eval_steps = 20
    gradient_accumulation_steps = 4
    learning_rate = 2e-4
elif RUN_MODE == 'mini':
    max_steps = 120
    num_train_epochs = 1
    save_steps = 60
    eval_steps = 30
    gradient_accumulation_steps = 8
    learning_rate = 1e-4
elif RUN_MODE == 'pilot':
    max_steps = 180
    num_train_epochs = 1
    save_steps = 60
    eval_steps = 30
    gradient_accumulation_steps = 16
    learning_rate = 1e-4
elif RUN_MODE == 'full':
    max_steps = -1
    num_train_epochs = 2
    save_steps = 500
    eval_steps = 250
    gradient_accumulation_steps = 16
    learning_rate = 6e-5
else:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')

import inspect

training_args_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=gradient_accumulation_steps,
    warmup_ratio=0.10,
    num_train_epochs=num_train_epochs,
    max_steps=max_steps,
    learning_rate=learning_rate,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_steps=eval_steps,
    eval_accumulation_steps=16,
    prediction_loss_only=True,
    save_total_limit=3,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=20260629,
    report_to='none',
    disable_tqdm=True,
    remove_unused_columns=False,
)

supported_training_args = set(inspect.signature(TrainingArguments.__init__).parameters)

# Transformers versions disagree on this name.
if 'eval_strategy' in supported_training_args:
    training_args_kwargs['eval_strategy'] = 'steps'
elif 'evaluation_strategy' in supported_training_args:
    training_args_kwargs['evaluation_strategy'] = 'steps'
else:
    print('Warning: this TrainingArguments version has no eval/evaluation strategy argument.')

# Some bleeding-edge Transformers builds removed/renamed this argument.
if 'save_strategy' in supported_training_args:
    training_args_kwargs['save_strategy'] = 'no'
else:
    print('Warning: save_strategy is not supported by this TrainingArguments version.')

if 'group_by_length' in supported_training_args:
    training_args_kwargs['group_by_length'] = GROUP_BY_LENGTH
else:
    print('Warning: group_by_length is not supported by this TrainingArguments version; continuing without it.')

# Drop any future-incompatible argument before constructing TrainingArguments.
training_args_kwargs = {
    key: value for key, value in training_args_kwargs.items()
    if key in supported_training_args
}
print('TrainingArguments keys:', sorted(training_args_kwargs))
training_args = TrainingArguments(**training_args_kwargs)

class LoRAAdapterCheckpointCallback(TrainerCallback):
    def __init__(self, tokenizer, output_dir, save_every_steps):
        self.tokenizer = tokenizer
        self.output_dir = Path(output_dir)
        self.save_every_steps = int(save_every_steps or 0)

    def on_step_end(self, args, state, control, **kwargs):
        control.should_save = False
        if self.save_every_steps <= 0 or state.global_step <= 0:
            return control
        if state.global_step % self.save_every_steps != 0:
            return control
        self._save_adapter(kwargs.get('model'), self.output_dir / f'lora-step-{state.global_step}')
        return control

    def _save_adapter(self, model_to_save, ckpt_dir):
        if model_to_save is None:
            return
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        model_to_save.save_pretrained(str(ckpt_dir))
        self.tokenizer.save_pretrained(str(ckpt_dir))
        print(f'Manual LoRA checkpoint saved: {ckpt_dir}')

class BestLoRACheckpointCallback(TrainerCallback):
    def __init__(self, tokenizer, output_dir):
        self.tokenizer = tokenizer
        self.output_dir = Path(output_dir)
        self.best_eval_loss = None

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        eval_loss = metrics.get('eval_loss')
        if eval_loss is None:
            return control
        if self.best_eval_loss is not None and eval_loss >= self.best_eval_loss:
            return control
        self.best_eval_loss = float(eval_loss)
        model_to_save = kwargs.get('model')
        best_dir = self.output_dir / 'best_lora_adapter'
        if model_to_save is not None:
            best_dir.mkdir(parents=True, exist_ok=True)
            model_to_save.save_pretrained(str(best_dir))
            self.tokenizer.save_pretrained(str(best_dir))
        payload = {
            'run_mode': RUN_MODE,
            'global_step': int(state.global_step),
            'epoch': float(state.epoch or 0.0),
            'best_eval_loss': self.best_eval_loss,
            'metrics': dict(metrics),
            'saved_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        self.output_dir.mkdir(parents=True, exist_ok=True)
        with (self.output_dir / 'best_eval_metrics.json').open('w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        print(f'Best LoRA adapter saved: {best_dir} eval_loss={self.best_eval_loss:.6f}')
        return control

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

def remove_notebook_progress_callbacks(trainer):
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if cb.__class__.__name__ != 'NotebookProgressCallback'
    ]
    return trainer

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with (OUTPUT_DIR / 'tokenization_stats.json').open('w', encoding='utf-8') as f:
    json.dump(tokenization_stats, f, ensure_ascii=False, indent=2)
print('Saved tokenization stats to:', OUTPUT_DIR / 'tokenization_stats.json')

trainer.add_callback(LoRAAdapterCheckpointCallback(tokenizer, OUTPUT_DIR, save_steps))
trainer.add_callback(BestLoRACheckpointCallback(tokenizer, OUTPUT_DIR))
remove_notebook_progress_callbacks(trainer)

print('Trainer ready.')
print('max_steps:', max_steps, 'eval_steps:', eval_steps, 'save_steps:', save_steps, 'grad_accum:', gradient_accumulation_steps, 'lr:', learning_rate)
print('effective_batch_size:', PER_DEVICE_TRAIN_BATCH_SIZE * gradient_accumulation_steps)


In [ ]:
remove_notebook_progress_callbacks(trainer)
trainer_stats = trainer.train()
print(trainer_stats)

import json
from datetime import datetime, timezone

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_stats_payload = {
    'run_mode': RUN_MODE,
    'model_name': MODEL_NAME,
    'model_preset': MODEL_PRESET,
    'use_native_system_role': USE_NATIVE_SYSTEM_ROLE,
    'max_seq_length': MAX_SEQ_LENGTH,
    'lora_config': {'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT, 'use_dora_requested': USE_DORA},
    'batch_config': {'train_batch': PER_DEVICE_TRAIN_BATCH_SIZE, 'eval_batch': PER_DEVICE_EVAL_BATCH_SIZE, 'group_by_length': GROUP_BY_LENGTH},
    'train_file': str(TRAIN_FILE),
    'val_file': str(VAL_FILE),
    'saved_at_utc': datetime.now(timezone.utc).isoformat(),
    'trainer_metrics': trainer_stats.metrics,
    'tokenization_stats': tokenization_stats,
}
with (OUTPUT_DIR / 'train_stats.json').open('w', encoding='utf-8') as f:
    json.dump(train_stats_payload, f, ensure_ascii=False, indent=2)
print('Saved train stats to:', OUTPUT_DIR / 'train_stats.json')


In [ ]:
import gc
import torch

remove_notebook_progress_callbacks(trainer)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

eval_metrics = trainer.evaluate()
print(eval_metrics)

import json
from datetime import datetime, timezone

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
eval_payload = {
    'run_mode': RUN_MODE,
    'model_name': MODEL_NAME,
    'model_preset': MODEL_PRESET,
    'use_native_system_role': USE_NATIVE_SYSTEM_ROLE,
    'max_seq_length': MAX_SEQ_LENGTH,
    'saved_at_utc': datetime.now(timezone.utc).isoformat(),
    'eval_metrics': eval_metrics,
}
with (OUTPUT_DIR / 'eval_metrics.json').open('w', encoding='utf-8') as f:
    json.dump(eval_payload, f, ensure_ascii=False, indent=2)
print('Saved eval metrics to:', OUTPUT_DIR / 'eval_metrics.json')


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUTPUT_DIR / 'lora_adapter'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'lora_adapter'))
print('Saved LoRA adapter to:', OUTPUT_DIR / 'lora_adapter')


## >>> COPY/UPDATE CELL 4 / 5: `quick-inference`
**QUICK INFERENCE - decode/pad fix burada**

Bu markdown başlığının hemen altındaki code cell’i komple kopyala/güncelle.


In [ ]:
# >>> COPY/UPDATE CELL 4 / 5: quick-inference
# === QUICK INFERENCE - decode/pad fix burada

from unsloth import FastModel
import json
from datetime import datetime, timezone

FastModel.for_inference(model)

PERFUME_CARDS = [
    {
        "name": "Aventus Cologne by Creed",
        "gender": "male",
        "accords": ["musky", "leather", "citrus", "fresh spicy", "aromatic", "smoky", "woody", "powdery"],
        "rating": "4.34/5 (3355 votes)",
        "seasons": "Spring, Summer",
        "time": "Day",
    },
    {
        "name": "Versace Pour Homme by Versace",
        "gender": "male",
        "accords": ["floral", "musky", "fresh spicy", "citrus", "aromatic", "green", "fresh"],
        "rating": "4.27/5 (21534 votes)",
        "seasons": "Spring, Summer",
        "time": "Day",
    },
    {
        "name": "Dior Sauvage Elixir by Dior",
        "gender": "male",
        "accords": ["aromatic", "warm spicy", "fresh spicy", "lavender", "woody", "sweet"],
        "rating": "4.23/5 (9800 votes)",
        "seasons": "Autumn, Winter",
        "time": "Night",
    },
    {
        "name": "Vanilla Woods by The 7 Virtues",
        "gender": "unisex",
        "accords": ["vanilla", "sweet", "woody", "amber", "powdery"],
        "rating": "4.05/5 (6700 votes)",
        "seasons": "Autumn, Winter",
        "time": "Day, Night",
    },
    {
        "name": "Prada L'Homme by Prada",
        "gender": "male",
        "accords": ["iris", "powdery", "clean", "woody", "amber", "neroli"],
        "rating": "4.33/5 (12000 votes)",
        "seasons": "Spring, Autumn",
        "time": "Day",
    },
]

TEST_CASES = [
    {
        "name": "fresh citrus summer cologne",
        "prompt": "Recommend a fresh citrus summer cologne for men.",
        "expect": "Prefer Aventus Cologne or Versace Pour Homme; avoid heavy winter/vanilla choices.",
        "excluded_accords": [],
    },
    {
        "name": "aventus less smoky",
        "prompt": "I want something like Aventus but less smoky.",
        "expect": "Aventus Cologne should be removed by strict filter because smoky is listed. Prefer Versace Pour Homme or Prada L'Homme. Do not include Vanilla Woods.",
        "excluded_accords": ["smoky"],
    },
    {
        "name": "clean office no vanilla",
        "prompt": "Recommend a clean office scent without vanilla.",
        "expect": "Vanilla Woods should be removed by strict filter. Prefer Prada L'Homme or Versace Pour Homme. Mention only listed accords.",
        "excluded_accords": ["vanilla"],
    },
]

SYSTEM = """You are ScentAI, a careful perfume recommendation assistant.
Your job is to recommend from the provided database context, not from memory.

GROUNDING CONTRACT:
- Recommend ONLY perfumes that appear inside [PERFUMES].
- Treat every perfume card as the complete source of truth for that perfume.
- Use only facts explicitly printed on that exact card.
- Do not use outside knowledge about perfume notes, flankers, brands, popularity, performance, or release details.

FIELD RULES:
- Accords and Notes are different fields.
- If a card has an Accords line, you may mention those listed accords as accords.
- You may mention notes ONLY when that exact card has a Notes, Top Notes, Middle Notes, or Base Notes line.
- If a card has no note field, never write "notes like", "note of", "with notes of", or any specific note names for that perfume.
- If a card has no rating, season, time, gender, longevity, sillage, or value field, do not infer it.

STRICT FILTERS:
- Respect [STRICT FILTERS] literally.
- Excluded notes, accords, or perfumes must be omitted, not ranked lower.
- For "less X" requests, avoid perfumes listing X when valid non-X alternatives exist.
- Do not claim "no listed X note" unless the card actually has a note field. Prefer "the card does not list X as an accord" only when X is absent from the Accords line.

ANSWER STYLE:
- Recommend 3-5 perfumes unless the user asks for a different number.
- For each perfume, give a short reason using only listed fields.
- Prefer wording like "listed accords include..." instead of inventing note-level detail.
- If the context has no strong match, say so honestly and recommend the closest grounded options.
- Respond in the same language as the user."""

def card_has_excluded_accord(card, excluded_accords):
    available = {accord.lower() for accord in card["accords"]}
    return any(term.lower() in available for term in excluded_accords)

def build_context(cards):
    chunks = []
    for card in cards:
        chunks.append("\n".join([
            f"{card['name']} - {card['gender']}",
            "Accords: " + ", ".join(card["accords"]),
            "Rating: " + card["rating"],
            f"Best Seasons: {card['seasons']} | Time: {card['time']}",
        ]))
    return "[PERFUMES]\n" + "\n\n".join(chunks) + "\n[/PERFUMES]"

def build_strict_filters(case):
    lines = ["[STRICT FILTERS]"]
    if case["excluded_accords"]:
        lines.append("Excluded accords: " + ", ".join(case["excluded_accords"]))
    else:
        lines.append("None")
    lines.append("[/STRICT FILTERS]")
    return "\n".join(lines)

PROCESSOR = tokenizer
TEXT_TOKENIZER = getattr(PROCESSOR, 'tokenizer', PROCESSOR)
if getattr(TEXT_TOKENIZER, 'pad_token_id', None) is None:
    TEXT_TOKENIZER.pad_token = TEXT_TOKENIZER.eos_token

device = next(model.parameters()).device

def generate_answer(case, max_new_tokens=260):
    allowed_cards = [card for card in PERFUME_CARDS if not card_has_excluded_accord(card, case["excluded_accords"])]
    forbidden_cards = [card for card in PERFUME_CARDS if card_has_excluded_accord(card, case["excluded_accords"])]
    context = build_context(allowed_cards)
    strict_filters = build_strict_filters(case)
    user_content = f"{strict_filters}\n\n{context}\n\n{case['prompt']}"
    if USE_NATIVE_SYSTEM_ROLE:
        messages = [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": user_content},
        ]
    else:
        messages = [{
            "role": "user",
            "content": f"{SYSTEM}\n\n{user_content}",
        }]
    inputs = PROCESSOR.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=not DISABLE_GEMMA_THINKING,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=TEXT_TOKENIZER.pad_token_id,
            eos_token_id=TEXT_TOKENIZER.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return decode_generated_tokens(generated), context, strict_filters, allowed_cards, forbidden_cards

inference_results = []
for case in TEST_CASES:
    decoded, context, strict_filters, allowed_cards, forbidden_cards = generate_answer(case)
    item = {
        "name": case["name"],
        "prompt": case["prompt"],
        "expected": case["expect"],
        "excluded_accords": case["excluded_accords"],
        "allowed_perfumes": [card["name"] for card in allowed_cards],
        "forbidden_perfumes": [card["name"] for card in forbidden_cards],
        "strict_filters": strict_filters,
        "context": context,
        "answer": decoded,
    }
    inference_results.append(item)
    print("\n" + "=" * 100)
    print(case["name"])
    print("Prompt:", case["prompt"])
    print("Expected:", case["expect"])
    print(strict_filters)
    print("--- Answer ---")
    print(decoded)



def extract_notes_like_terms(line):
    import re
    match = re.search(r"notes like\s+(.+?)(?:,?\s+and\s+|\s+and\s+)?(?:a\s+|an\s+)?(?:day|night|spring|summer|autumn|winter|wear|profile|suitability|rating|accord|accords|$)", line, re.I)
    if not match:
        return []
    text = re.split(r"\band\b|;", match.group(1))[0]
    return [part.strip(" .,-") for part in text.split(",") if part.strip(" .,-")]

def find_unsupported_note_claims(item):
    # Quick-test cards intentionally omit Notes lines. If the answer says "notes like ...",
    # that is unsupported unless the card explicitly has a notes field later.
    answer = item["answer"]
    allowed_cards = [card for card in PERFUME_CARDS if card["name"] in item.get("allowed_perfumes", [])]
    current_card = None
    findings = []
    for raw_line in answer.splitlines():
        line = raw_line.strip()
        lowered = line.lower()
        for card in allowed_cards:
            if card["name"].lower() in lowered:
                current_card = card
                break
        if "notes like" not in lowered:
            continue
        claimed = extract_notes_like_terms(line)
        if current_card is None:
            findings.append({"name": None, "claimed_notes": claimed, "reason": "notes-like claim has no matched perfume"})
            continue
        supported_notes = {note.lower() for note in current_card.get("notes", [])}
        if not supported_notes:
            findings.append({"name": current_card["name"], "claimed_notes": claimed, "reason": "perfume card has no Notes line"})
            continue
        unsupported = [note for note in claimed if note.lower() not in supported_notes]
        if unsupported:
            findings.append({"name": current_card["name"], "claimed_notes": claimed, "unsupported_notes": unsupported, "reason": "claimed note not present in perfume card"})
    return findings

def score_case_result(item):
    answer_lower = item["answer"].lower()
    allowed = item.get("allowed_perfumes", [])
    forbidden = item.get("forbidden_perfumes", [])
    mentioned_allowed = [name for name in allowed if name.lower() in answer_lower]
    mentioned_forbidden = [name for name in forbidden if name.lower() in answer_lower]
    context_leaks = []
    for card in PERFUME_CARDS:
        if card["name"] not in allowed:
            continue
        leaks = sorted(set(a.lower() for a in card["accords"]) & set(a.lower() for a in item.get("excluded_accords", [])))
        if leaks:
            context_leaks.append({"name": card["name"], "excluded_accords": leaks})
    unsupported_note_claims = find_unsupported_note_claims(item)
    hard_fail_reasons = []
    if mentioned_forbidden:
        hard_fail_reasons.append("Forbidden perfume appeared in the answer.")
    if context_leaks:
        hard_fail_reasons.append("Strict filter failed before generation; excluded accord remained in context.")
    if unsupported_note_claims:
        hard_fail_reasons.append("Answer claimed notes not shown in the matching perfume card.")
    warnings = []
    if not mentioned_allowed:
        warnings.append("No allowed perfume name was detected in the answer.")
    return {
        "name": item["name"],
        "pass": not hard_fail_reasons,
        "hard_fail_reasons": hard_fail_reasons,
        "warnings": warnings,
        "mentioned_allowed_perfumes": mentioned_allowed,
        "mentioned_forbidden_perfumes": mentioned_forbidden,
        "context_filter_leaks": context_leaks,
        "unsupported_note_claims": unsupported_note_claims,
    }

grounding_report = {
    "case_count": len(inference_results),
    "cases": [score_case_result(item) for item in inference_results],
}
grounding_report["pass_count"] = sum(1 for item in grounding_report["cases"] if item["pass"])
grounding_report["hard_failure_count"] = grounding_report["case_count"] - grounding_report["pass_count"]
grounding_report["pass_rate"] = grounding_report["pass_count"] / max(grounding_report["case_count"], 1)
print("\nGrounding report:")
print(json.dumps(grounding_report, ensure_ascii=False, indent=2))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
inference_payload = {
    "run_mode": RUN_MODE,
    "model_name": MODEL_NAME,
    "model_preset": MODEL_PRESET,
    "max_seq_length": MAX_SEQ_LENGTH,
    "use_native_system_role": USE_NATIVE_SYSTEM_ROLE,
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "system": SYSTEM,
    "grounding_report": grounding_report,
    "results": inference_results,
}
json_path = OUTPUT_DIR / "quick_inference_results.json"
txt_path = OUTPUT_DIR / "quick_inference_results.txt"
grounding_path = OUTPUT_DIR / "grounding_report.json"
with json_path.open("w", encoding="utf-8") as f:
    json.dump(inference_payload, f, ensure_ascii=False, indent=2)
with grounding_path.open("w", encoding="utf-8") as f:
    json.dump(grounding_report, f, ensure_ascii=False, indent=2)

lines = [
    f"run_mode: {RUN_MODE}",
    f"model_name: {MODEL_NAME}",
    f"model_preset: {MODEL_PRESET}",
    f"use_native_system_role: {USE_NATIVE_SYSTEM_ROLE}",
    f"grounding_pass_rate: {grounding_report['pass_rate']:.3f}",
    f"saved_at_utc: {inference_payload['saved_at_utc']}",
    "",
]
for item in inference_results:
    lines.extend([
        "=" * 100,
        item["name"],
        f"Prompt: {item['prompt']}",
        f"Expected: {item['expected']}",
        f"Allowed: {', '.join(item['allowed_perfumes'])}",
        f"Forbidden: {', '.join(item['forbidden_perfumes']) if item['forbidden_perfumes'] else 'None'}",
        item["strict_filters"],
        "--- Context ---",
        item["context"],
        "--- Answer ---",
        item["answer"],
        "",
    ])
txt_path.write_text("\n".join(lines), encoding="utf-8")
print("\nSaved quick inference JSON to:", json_path)
print("Saved quick inference TXT to:", txt_path)
print("Saved grounding report to:", grounding_path)


## >>> COPY/UPDATE EVAL CELL: `scentai-eval-runner`

Run this after the LoRA/model is loaded to generate fixed eval outputs.


In [ ]:
# Paste this cell into Colab after the fine-tuned model/LoRA is loaded.
# It expects these notebook variables/functions to already exist:
# model, tokenizer, FastModel, PROCESSOR, TEXT_TOKENIZER, decode_generated_tokens,
# USE_NATIVE_SYSTEM_ROLE, DISABLE_GEMMA_THINKING, device, OUTPUT_DIR.

import json
import os
from datetime import datetime, timezone
from pathlib import Path

EVAL_SET_PATH = PROJECT_DIR / "train_set" / "eval" / "scentai_eval_v2.jsonl"
EVAL_RUN_DIR = PROJECT_DIR / "train_set" / "eval" / "runs"
EVAL_RUN_NAME = f"{RUN_MODE}_{MODEL_PRESET}_eval_v2_strictprompt_v2"
EVAL_OUTPUTS_PATH = EVAL_RUN_DIR / f"{EVAL_RUN_NAME}_outputs.jsonl"
EVAL_METADATA_PATH = EVAL_RUN_DIR / f"{EVAL_RUN_NAME}_metadata.json"
EVAL_LIMIT = None  # set to 20 for a quick smoke pass
EVAL_START_INDEX = 0  # 0-based inclusive; use 0, 40, 80, 120 for chunks
EVAL_END_INDEX = None  # exclusive; use 40, 80, 120, 160 for chunks
EVAL_MAX_NEW_TOKENS = 300
EVAL_RESUME = False

EVAL_SYSTEM = """You are ScentAI, a careful perfume recommendation assistant.
Your job is to recommend from the provided database context, not from memory.

GROUNDING CONTRACT:
- Recommend ONLY perfumes that appear inside [PERFUMES].
- Treat every perfume card as the complete source of truth for that perfume.
- Use only facts explicitly printed on that exact card.
- Do not use outside knowledge about perfume notes, flankers, brands, popularity, performance, or release details.

FIELD RULES:
- Accords and Notes are different fields.
- If a card has an Accords line, you may mention those listed accords as accords.
- You may mention notes ONLY when that exact card has a Notes, Top Notes, Middle Notes, or Base Notes line.
- If a card has no note field, never write "notes like", "note of", "with notes of", or any specific note names for that perfume.
- If a card has no rating, season, time, gender, longevity, sillage, or value field, do not infer it.

DATABASE LOOKUP MODE:
- If the user asks what the database says, asks for a database record, or asks for exact perfume information, copy fields exactly from the card.
- Do not convert `0.00/5 (0 votes)` to `N/A`.
- Do not simplify accords such as `white floral` into `floral`.
- Do not omit notes from a single-perfume record unless you explicitly say you are summarizing.

STRICT FILTERS:
- Respect [STRICT FILTERS] literally.
- Excluded notes, accords, or perfumes must be omitted, not ranked lower.
- If [STRICT FILTERS] lists forbidden perfumes, do not mention them in recommendations, reasons, or Best pick.
- For "less X" requests, avoid perfumes listing X when valid non-X alternatives exist.
- Do not claim "no listed X note" unless the card actually has a note field. Prefer "the card does not list X as an accord" only when X is absent from the Accords line.
- If no safe perfume remains after strict filters, say that clearly instead of recommending a forbidden perfume.

ANSWER STYLE:
- Recommend 3-5 perfumes unless the user asks for a different number.
- For each perfume, give a short reason using only listed fields.
- Prefer wording like "listed accords include..." instead of inventing note-level detail.
- If the context has no strong match, say so honestly and recommend the closest grounded options.
- Respond in the same language as the user."""


def read_eval_cases(path):
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def find_output_candidates(primary_path):
    candidates = [primary_path]
    unique = []
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key not in seen and candidate.exists():
            unique.append(candidate)
            seen.add(key)
    return unique


def read_completed_outputs(paths):
    if not EVAL_RESUME:
        return {}, {"sources": [], "malformed_lines": 0}
    completed = {}
    malformed_lines = 0
    sources = []
    for path in paths:
        sources.append(str(path))
        with path.open(encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    malformed_lines += 1
                    print(f"Skipping malformed JSONL line: {path}:{line_number}")
                    continue
                case_id = row.get("id")
                if case_id:
                    completed[case_id] = row
    return completed, {"sources": sources, "malformed_lines": malformed_lines}


def write_completed_snapshot(path, completed, cases):
    if not completed:
        return
    ordered_ids = [case["id"] for case in cases]
    with path.open("w", encoding="utf-8") as handle:
        for case_id in ordered_ids:
            row = completed.get(case_id)
            if row:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def generate_eval_answer(case):
    user = inject_eval_strict_filters(case["user"].strip(), case.get("checks", {}))
    if USE_NATIVE_SYSTEM_ROLE:
        messages = [{"role": "system", "content": EVAL_SYSTEM}, {"role": "user", "content": user}]
    else:
        content = f"{SYSTEM_OPEN}\n{EVAL_SYSTEM}\n{SYSTEM_CLOSE}\n\n{user}"
        messages = [{"role": "user", "content": content}]

    inputs = PROCESSOR.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=not DISABLE_GEMMA_THINKING,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=EVAL_MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=TEXT_TOKENIZER.pad_token_id,
            eos_token_id=TEXT_TOKENIZER.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return decode_generated_tokens(generated)


def inject_eval_strict_filters(user, checks):
    block = build_eval_strict_filter_block(checks)
    if block is None or "[STRICT FILTERS]" in user:
        return user
    marker = "[/PERFUMES]"
    if marker in user:
        return user.replace(marker, marker + "\n\n" + block, 1)
    return block + "\n\n" + user


def build_eval_strict_filter_block(checks):
    excluded_terms = checks.get("excluded_terms", []) or []
    forbidden_perfumes = checks.get("forbidden_perfumes", []) or []
    if not excluded_terms and not forbidden_perfumes:
        return None
    lines = ["[STRICT FILTERS]"]
    if excluded_terms:
        lines.append("Excluded terms: " + ", ".join(excluded_terms))
    if forbidden_perfumes:
        lines.append("Forbidden perfumes: " + "; ".join(forbidden_perfumes))
    lines.append("You must not recommend forbidden perfumes. If no allowed option fits, say no safe match.")
    lines.append("[/STRICT FILTERS]")
    return "\n".join(lines)


FastModel.for_inference(model)
cases = read_eval_cases(EVAL_SET_PATH)
if EVAL_LIMIT is not None:
    cases = cases[:EVAL_LIMIT]
if EVAL_END_INDEX is not None or EVAL_START_INDEX:
    cases = cases[EVAL_START_INDEX:EVAL_END_INDEX]

EVAL_OUTPUTS_PATH.parent.mkdir(parents=True, exist_ok=True)
candidate_paths = find_output_candidates(EVAL_OUTPUTS_PATH)
completed, resume_info = read_completed_outputs(candidate_paths)
write_completed_snapshot(EVAL_OUTPUTS_PATH, completed, cases)
mode = "a" if completed and EVAL_RESUME else "w"

metadata = {
    "eval_set_path": str(EVAL_SET_PATH),
    "eval_outputs_path": str(EVAL_OUTPUTS_PATH),
    "eval_limit": EVAL_LIMIT,
    "eval_start_index": EVAL_START_INDEX,
    "eval_end_index": EVAL_END_INDEX,
    "eval_max_new_tokens": EVAL_MAX_NEW_TOKENS,
    "eval_resume": EVAL_RESUME,
    "case_count": len(cases),
    "already_completed": len(completed),
    "resume_sources": resume_info["sources"],
    "resume_malformed_lines": resume_info["malformed_lines"],
    "model_name": MODEL_NAME,
    "model_preset": MODEL_PRESET,
    "run_mode": RUN_MODE,
    "output_dir": str(OUTPUT_DIR),
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
}
EVAL_METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
with EVAL_METADATA_PATH.open("a", encoding="utf-8") as meta_handle:
    meta_handle.flush()
    os.fsync(meta_handle.fileno())
print("Eval resume sources:", resume_info["sources"])
print("Eval malformed lines skipped:", resume_info["malformed_lines"])
print("Eval already completed:", len(completed), "/", len(cases))

with EVAL_OUTPUTS_PATH.open(mode, encoding="utf-8") as handle:
    for index, case in enumerate(cases, 1):
        if case["id"] in completed:
            print(f"[{index}/{len(cases)}] skip {case['id']} {case['category']}")
            continue
        answer = generate_eval_answer(case)
        row = {
            "id": case["id"],
            "category": case["category"],
            "mode": case.get("mode"),
            "difficulty": case.get("difficulty"),
            "tags": case.get("tags", []),
            "answer": answer,
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
        print(f"[{index}/{len(cases)}] {case['id']} {case['category']} chars={len(answer)}")

print("Saved eval outputs to:", EVAL_OUTPUTS_PATH)
print("Saved eval metadata to:", EVAL_METADATA_PATH)


In [ ]:
print('FULL RUN COMPLETE')
print('LoRA adapter:', OUTPUT_DIR / 'lora_adapter')
print('Train stats:', OUTPUT_DIR / 'train_stats.json')
print('Eval metrics:', OUTPUT_DIR / 'eval_metrics.json')
print('Quick inference:', OUTPUT_DIR / 'quick_inference_results.json')
print('Eval outputs:', EVAL_OUTPUTS_PATH)
print('Eval metadata:', EVAL_METADATA_PATH)
